In [ ]:
!pip install "numpy==1.26.4" paddlepaddle-gpu "paddleocr==2.8.1" opencv-python "requests==2.32.4"

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, shutil, zipfile, json, re, random, collections, logging
from PIL import Image
import cv2
from paddleocr import PaddleOCR
from tqdm.auto import tqdm
from pathlib import Path
import re as _re
import torch
from torch.utils.data import Dataset, DataLoader

from typing import List, Dict, Any

## Mount Drive, set paths, install

In [ ]:
PROJECT_NAME = 'VU_DL_Team_Project'
DRIVE_ROOT   = Path('/content/drive/MyDrive') / PROJECT_NAME if IN_COLAB \
               else Path('./drive_local') / PROJECT_NAME

DATA_SUBDIR = 'pii_v5'
DATA_DRIVE   = DRIVE_ROOT / 'data' / DATA_SUBDIR
OUTPUTS      = DRIVE_ROOT / 'outputs'
ADAPTER_DIR  = OUTPUTS / 'lora_adapters_v5' / 'final'
CKPT_DIR     = OUTPUTS / 'checkpoints'

# Local fast scratch for images
LOCAL_SCRATCH = Path('/content/scratch') if IN_COLAB else Path('./scratch')
DATA_LOCAL    = LOCAL_SCRATCH / DATA_SUBDIR

for p in [DATA_DRIVE, OUTPUTS, ADAPTER_DIR, CKPT_DIR, DATA_LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

# Required files
KEEP_FILES = ['train.jsonl', 'val.jsonl', 'test.jsonl', 'images.zip']

if IN_COLAB:
    # Check if all required files exist
    missing_files = [f for f in KEEP_FILES if not (DATA_DRIVE / f).exists()]

    if missing_files:
        print(f"Missing files: {missing_files}. Starting target-only download...")

        # 1. TELL THE CODE THE EXACT FILE IDs (Don't use the folder ID anymore)
        # To get these, right-click the file in Drive -> Share -> Copy Link.
        # The ID is the long string of letters and numbers in the middle of the link.
        FILE_IDS = {
            'train.jsonl': '1e4qNb05zbcUEOLmsBJN788n0fw9fWr0r',
            'val.jsonl':   '1pFH80PT8NdRTEfuznVu48IKjRooZ_oWg',
            'test.jsonl':  '1a7oYHG-rDPTOECkr08qBSXGU-Wf2FPxT',
            'images.zip':  '1AV_SoMJJeUpjV3UZyIofapLzPCrL6E13'
        }

        # 2. Loop through and download ONLY the missing files
        for file_name in missing_files:
            file_id = FILE_IDS.get(file_name)
            if file_id and not file_id.startswith('REPLACE_'):
                print(f"Downloading {file_name}...")
                destination = DATA_DRIVE / file_name

                # Download the single file directly
                !gdown {file_id} -O {str(destination)}
            else:
                print(f"⚠️ Skipping {file_name}: Valid File ID not provided yet.")

        print("✅ Target download check complete!")
    else:
        print("✅ All target files (train, val, test, images.zip) already exist in Drive. Skipping download.")

## Sync data to local disk (fast training reads)

In [6]:
# Copy JSONL files (tiny)
for split in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    src = DATA_DRIVE / split
    dst = DATA_LOCAL / split
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f'Copied {split}')

# Unzip images to local disk (only once per session)
images_local = DATA_LOCAL / 'images'

# Ensure the target directory for unzipping exists
images_local.mkdir(parents=True, exist_ok=True)

# Check if images are already unzipped into the correct local directory
if not any(images_local.iterdir()):
    images_zip = DATA_DRIVE / 'images.zip'
    assert images_zip.exists(), f'Missing {images_zip} — re-run notebook 01 section 9'
    print(f'Unzipping {images_zip.stat().st_size/1e6:.1f} MB to local disk...')
    with zipfile.ZipFile(images_zip) as zf:
        # Extract contents directly into the pre-created images_local directory
        zf.extractall(images_local)
    print(f'✅ {len(list(images_local.iterdir()))} images unzipped')
else:
    print(f'✅ images already on local disk: {len(list(images_local.iterdir()))}')

Unzipping 5360.7 MB to local disk...
✅ 10614 images unzipped


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Sanity check — object-count & label distribution

In [7]:
for split in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    p = DATA_LOCAL / split
    if not p.exists():
        print(f'(skipping {split} — not found)'); continue
    rows = [json.loads(l) for l in open(p)]
    obj_counts = collections.Counter(len(r['objects']) for r in rows)
    labels = collections.Counter(o['label'] for r in rows for o in r['objects'])
    total_screens = sum(obj_counts.values())
    over_cap = sum(v for k, v in obj_counts.items() if k > 12)
    print(f'\n── {split}  ({total_screens} screens) ──')
    print('  objects/screen:', dict(sorted(obj_counts.items())))
    print(f'  screens with >12 boxes (over current MAX_OBJECTS): {over_cap}')
    print('  label distribution:', dict(labels.most_common()))


── train.jsonl  (6999 screens) ──
  objects/screen: {0: 913, 1: 3629, 2: 1174, 3: 732, 4: 209, 5: 109, 6: 120, 7: 29, 8: 18, 9: 39, 10: 7, 11: 5, 12: 5, 14: 2, 15: 5, 16: 2, 18: 1}
  screens with >12 boxes (over current MAX_OBJECTS): 10
  label distribution: {'address': 3135, 'transaction_amount': 2316, 'email_address': 1975, 'other_sensitive': 1260, 'date_of_birth': 884, 'phone_number': 816, 'username': 570, 'full_name': 201, 'account_balance': 153}

── val.jsonl  (875 screens) ──
  objects/screen: {0: 114, 1: 430, 2: 162, 3: 91, 4: 34, 5: 8, 6: 24, 7: 2, 8: 3, 9: 5, 11: 1, 13: 1}
  screens with >12 boxes (over current MAX_OBJECTS): 1
  label distribution: {'address': 373, 'transaction_amount': 323, 'email_address': 240, 'other_sensitive': 183, 'date_of_birth': 115, 'phone_number': 109, 'username': 66, 'full_name': 27, 'account_balance': 18}

── test.jsonl  (882 screens) ──
  objects/screen: {0: 115, 1: 445, 2: 171, 3: 87, 4: 29, 5: 13, 6: 13, 7: 2, 8: 1, 9: 1, 10: 1, 11: 1, 12: 2, 1

## Stage 1. OCR extracting bbox

In [8]:
# Suppress PaddleOCR's massive wall of text logs
logging.getLogger("ppocr").setLevel(logging.ERROR)

class ScreenExtractor:
    def __init__(self, use_gpu: bool = True) -> None:
        print("⏳ Initializing PaddleOCR Engine...")
        self.ocr = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=use_gpu, show_log=False)
        print("✅ Stage 1 Extractor Ready!")

    def extract_text_elements(self, image_path: str) -> List[Dict[str, Any]]:
        extracted_elements = []

        # SAFETY CHECK: Stop silently failing if the image is missing!
        if not os.path.exists(image_path):
            print(f"❌ ERROR: Image not found at {image_path}")
            return extracted_elements

        results = self.ocr.ocr(image_path, cls=True)

        if not results or not results[0]:
            return extracted_elements

        for line in results[0]:
            polygon, (text, confidence) = line

            x_coords = [point[0] for point in polygon]
            y_coords = [point[1] for point in polygon]

            x1, x2 = min(x_coords), max(x_coords)
            y1, y2 = min(y_coords), max(y_coords)

            extracted_elements.append({
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
                "text": text,
                "confidence": round(float(confidence), 4)
            })

        return extracted_elements

# Initialize the extractor
extractor = ScreenExtractor(use_gpu=True)

⏳ Initializing PaddleOCR Engine...
download https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_det_infer.tar to /root/.paddleocr/whl/det/en/en_PP-OCRv3_det_infer/en_PP-OCRv3_det_infer.tar


100%|██████████| 4.00M/4.00M [00:00<00:00, 5.72MiB/s]


download https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_rec_infer.tar to /root/.paddleocr/whl/rec/en/en_PP-OCRv4_rec_infer/en_PP-OCRv4_rec_infer.tar


100%|██████████| 10.2M/10.2M [00:00<00:00, 11.8MiB/s]


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:00<00:00, 3.63MiB/s]


✅ Stage 1 Extractor Ready!


In [9]:
def run_stage_1_pipeline(split_name: str):
    input_path = DATA_LOCAL / split_name

    if not input_path.exists():
        print(f"⚠️ Skipping {split_name} — not found.")
        return

    # Create a new filename for the OCR-augmented data
    output_name = split_name.replace('.jsonl', '_ocr.jsonl')
    output_path = DATA_LOCAL / output_name

    # Also save a backup directly to Google Drive so you don't have to run OCR again tomorrow!
    drive_backup_path = DATA_DRIVE / output_name

    with open(input_path, 'r') as f_in, open(output_path, 'w') as f_out:
        lines = f_in.readlines()

        for line in tqdm(lines, desc=f"🔍 OCR Processing {split_name}"):
            row = json.loads(line)

            # Construct the absolute path to the image in the local scratch folder
            img_path = str(DATA_LOCAL / row['image'])

            # 1. Run PaddleOCR
            ocr_results = extractor.extract_text_elements(img_path)

            # 2. Inject the results into the row dictionary
            row['ocr_elements'] = ocr_results

            # 3. Write to the new JSONL file
            f_out.write(json.dumps(row) + '\n')

    # Copy the finished file back to Google Drive for permanent storage
    shutil.copy(output_path, drive_backup_path)
    print(f"✅ Finished {split_name}. Saved to local and backed up to Drive at: {drive_backup_path.name}\n")

# Run the pipeline over all splits
for split in ['test.jsonl', 'val.jsonl', 'train.jsonl']:
    run_stage_1_pipeline(split)

🔍 OCR Processing test.jsonl:   0%|          | 0/882 [00:00<?, ?it/s]

✅ Finished test.jsonl. Saved to local and backed up to Drive at: test_ocr.jsonl



🔍 OCR Processing val.jsonl:   0%|          | 0/875 [00:00<?, ?it/s]

✅ Finished val.jsonl. Saved to local and backed up to Drive at: val_ocr.jsonl



🔍 OCR Processing train.jsonl:   0%|          | 0/6999 [00:00<?, ?it/s]

✅ Finished train.jsonl. Saved to local and backed up to Drive at: train_ocr.jsonl



## Stage 2. Finetuning Gemma 4

### Preparing dataset for Gemma4

In [ ]:
# Copy Gemma JSONL files (tiny)
for split in ['gemma_train.jsonl', 'gemma_val.jsonl', 'gemma_test.jsonl']:
    src = DATA_DRIVE / split
    dst = DATA_LOCAL / split
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f'Copied {split}')

Copied gemma_train.jsonl
Copied gemma_val.jsonl
Copied gemma_test.jsonl


In [ ]:
# Copy Gemma JSONL files (tiny)
for split in ['train_ocr.jsonl', 'val_ocr.jsonl', 'test_ocr.jsonl']:
    src = DATA_DRIVE / split
    dst = DATA_LOCAL / split
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)
        print(f'Copied {split}')

Copied train_ocr.jsonl
Copied val_ocr.jsonl
Copied test_ocr.jsonl


In [10]:
_EMAIL_RE = _re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
_PHONE_RE = _re.compile(r'^[\+\d][\d\s\-\(\)]{6,}$')

def calculate_iou(box1, box2):
    """Calculates Intersection over Union for two [x1, y1, x2, y2] boxes."""
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area if union_area > 0 else 0.0

def is_contained(small, big, frac=0.7):
    """True if `frac` of `small`'s area is inside `big`."""
    ix1, iy1 = max(small[0], big[0]), max(small[1], big[1])
    ix2, iy2 = min(small[2], big[2]), min(small[3], big[3])
    if ix2 <= ix1 or iy2 <= iy1:
        return False
    inter = (ix2 - ix1) * (iy2 - iy1)
    small_area = max(1, (small[2]-small[0]) * (small[3]-small[1]))
    return inter / small_area >= frac

def clean_ocr_text(text: str) -> str:
    """Remove markdown link decorations OCR sometimes injects."""
    t = text.strip()
    # Match [visible](scheme:anything) with flexible whitespace
    m = re.match(r'^\[\s*([^\]]+?)\s*\]\s*\(\s*(?:mailto:|tel:|https?://)[^)]*\)\s*$', t)
    if m:
        return m.group(1).strip()
    # Also handle the case where text contains the markdown mid-string (rare)
    t = re.sub(r'\[([^\]]+)\]\((?:mailto:|tel:|https?://)[^)]*\)', r'\1', t)
    return t

def assign_label(ocr_box, gt_objects, ocr_text=None, iou_thr=0.3, contain_frac=0.7):
    """Best-effort label assignment: IoU OR containment."""
    best_iou, best_label = 0.0, None
    for gt in gt_objects:
        iou = calculate_iou(ocr_box, gt['bbox'])
        if iou >= iou_thr and iou > best_iou:
            best_iou, best_label = iou, gt['label']
        elif is_contained(ocr_box, gt['bbox'], frac=contain_frac):
            # Containment fallback — catches multi-line address pieces
            if best_label is None:
                best_label = gt['label']
    # Content-based override: if the text obviously looks like an email
    # or phone, prefer that label over the field's UI role.
    if best_label is not None and ocr_text:
        t = ocr_text.strip()
        if _EMAIL_RE.match(t):
            return 'email_address'
        if _PHONE_RE.match(t) and best_label != 'phone_number':
            return 'phone_number'
    return best_label

def build_finetuning_dataset_v2(input_jsonl, output_jsonl, drive_output_jsonl=None,
                                 images_dir=None):
    """V2: includes layout (bbox coords) in prompt, uses containment fallback.

    Args:
        images_dir: Path to the directory holding the screen images. If None,
                    falls back to DATA_LOCAL (the standard layout where
                    row['image'] is relative to DATA_LOCAL).
    """
    if images_dir is None:
        images_dir = DATA_LOCAL

    n_in, n_out, n_no_ocr, n_no_image = 0, 0, 0, 0

    with open(input_jsonl, 'r') as f_in, open(output_jsonl, 'w') as f_out:
        for line in f_in:
            n_in += 1
            data = json.loads(line)
            gt_objects = data.get('objects', [])
            ocr_elements = data.get('ocr_elements', [])
            if not ocr_elements:
                n_no_ocr += 1
                continue

            # ── NEW: read the actual image size for this screen ─────────────
            img_path = images_dir / data['image']
            if not img_path.exists():
                n_no_image += 1
                continue
            try:
                with Image.open(img_path) as im:
                    img_w, img_h = im.size      # (width, height) in pixels
            except Exception as e:
                print(f"⚠️ Could not open {img_path}: {e}")
                n_no_image += 1
                continue
            # ────────────────────────────────────────────────────────────────

            # Sort OCR boxes top-to-bottom, then left-to-right (reading order)
            # Bucket Y in ~2% rows so near-horizontal boxes group as one line
            row_bucket = max(1, img_h // 50)
            ocr_sorted = sorted(
                ocr_elements,
                key=lambda e: (e['bbox'][1] // row_bucket, e['bbox'][0])
            )

            elements_with_layout = []
            pii_answers = {}
            for i, ocr in enumerate(ocr_sorted):
                x1, y1, x2, y2 = ocr['bbox']
                text_clean = clean_ocr_text(ocr['text'])
                # Normalize to a 0–1000 grid (so prompts are size-invariant)
                nx = int(1000 * (x1 + x2) / 2 / max(1, img_w))
                ny = int(1000 * (y1 + y2) / 2 / max(1, img_h))
                nx = min(999, max(0, nx))      # clamp in case OCR returns out-of-bounds
                ny = min(999, max(0, ny))
                tag = f'[{i}@{nx},{ny}]'
                elements_with_layout.append(f'{tag} "{text_clean}"')

                label = assign_label(ocr['bbox'], gt_objects, ocr_text=text_clean)
                if label is not None:
                    pii_answers[tag] = label

            elements_str = '\n'.join(elements_with_layout)
            prompt = (
                "You are a privacy auditor. Below are text elements from a mobile screenshot. "
                "Each line is formatted as [index@x,y] \"text\" where x,y is the element's center "
                "on a 1000x1000 normalized grid (top-left = 0,0).\n\n"
                "Classify each element that contains personally identifiable information (PII) "
                "into one of: email_address, phone_number, full_name, username, address, "
                "date_of_birth, account_balance, transaction_amount, profile_photo, other_sensitive.\n\n"
                "Return ONLY a JSON object mapping the [index@x,y] tag to its label. "
                "Omit non-PII elements. Example: {\"[3@500,200]\": \"email_address\"}.\n\n"
                "ELEMENTS:\n" + elements_str + "\n\nJSON:"
            )

            answer = json.dumps(pii_answers)

            f_out.write(json.dumps({
                "screen_id": data["screen_id"],
                "prompt": prompt,
                "answer": answer
            }) + '\n')
            n_out += 1

    print(f"✅ {output_jsonl.name}: {n_out}/{n_in} written  "
          f"(skipped: {n_no_ocr} no-OCR, {n_no_image} missing-image)")
    if drive_output_jsonl:
        shutil.copy(output_jsonl, drive_output_jsonl)

In [11]:
build_finetuning_dataset_v2(
    input_jsonl=DATA_LOCAL / "train_ocr.jsonl",
    output_jsonl=DATA_LOCAL / "gemma_train.jsonl",
    drive_output_jsonl=DATA_DRIVE / "gemma_train.jsonl",
)
build_finetuning_dataset_v2(
    input_jsonl=DATA_LOCAL / "val_ocr.jsonl",
    output_jsonl=DATA_LOCAL / "gemma_val.jsonl",
    drive_output_jsonl=DATA_DRIVE / "gemma_val.jsonl",
)
build_finetuning_dataset_v2(
    input_jsonl=DATA_LOCAL / "test_ocr.jsonl",
    output_jsonl=DATA_LOCAL / "gemma_test.jsonl",
    drive_output_jsonl=DATA_DRIVE / "gemma_test.jsonl",
)

✅ gemma_train.jsonl: 6999/6999 written  (skipped: 0 no-OCR, 0 missing-image)
✅ gemma_val.jsonl: 875/875 written  (skipped: 0 no-OCR, 0 missing-image)
✅ gemma_test.jsonl: 882/882 written  (skipped: 0 no-OCR, 0 missing-image)


In [12]:
with open(DATA_LOCAL / "gemma_train.jsonl") as f:
    row = json.loads(next(f))
print(row['prompt'][:1500])
print('---ANSWER---')
print(row['answer'])

You are a privacy auditor. Below are text elements from a mobile screenshot. Each line is formatted as [index@x,y] "text" where x,y is the element's center on a 1000x1000 normalized grid (top-left = 0,0).

Classify each element that contains personally identifiable information (PII) into one of: email_address, phone_number, full_name, username, address, date_of_birth, account_balance, transaction_amount, profile_photo, other_sensitive.

Return ONLY a JSON object mapping the [index@x,y] tag to its label. Omit non-PII elements. Example: {"[3@500,200]": "email_address"}.

ELEMENTS:
[0@924,17] "8:57"
[1@512,111] "Sbabycenter"
[2@498,164] "My Pregnancy Today"
[3@242,218] "An unknown error occurred"
[4@341,289] "Log in to your BabyCenter account"
[5@285,334] "appcrawler6@gmail.com"
[6@861,381] "Show"
[7@231,441] "I forgot my password"
[8@500,501] "Log in and get started"
[9@497,556] "Not yet a BabyCenter member?"
[10@499,583] "Sign up"
[11@132,908] "Privacy Policy"
[12@870,906] "Terms of Use

In [14]:
file_path = Path('/content/scratch/pii_v5/train_ocr.jsonl')

empty_count = 0
success_count = 0

with open(file_path, 'r') as f:
    for line in f:
        row = json.loads(line)
        if len(row.get('ocr_elements', [])) == 0:
            empty_count += 1
        else:
            success_count += 1

print(f"📊 OCR Extraction Results for {file_path.name}:")
print(f"✅ Successful Screens (Text Found): {success_count}")
print(f"❌ Empty Screens (No Text Found): {empty_count}")

📊 OCR Extraction Results for train_ocr.jsonl:
✅ Successful Screens (Text Found): 6999
❌ Empty Screens (No Text Found): 0
